# LIMPIEZA DE DATOS
## PASO 1: Importacion y carga inicial


In [1]:
import pandas as pd
df_covid = pd.read_csv("positivos_covid 3.csv", sep=";")
df_covid.head()

,FECHA_CORTE,DEPARTAMENTO,PROVINCIA,DISTRITO,METODODX,EDAD,SEXO,FECHA_RESULTADO,UBIGEO,id_persona
0,20241203,TUMBES,TUMBES,TUMBES,AG,46.0,FEMENINO,20221207.0,240101.0,203499.0
1,20241203,LIMA,LIMA,JESUS MARIA,AG,69.0,FEMENINO,20230822.0,150113.0,221397.0
2,20241203,SAN MARTIN,MOYOBAMBA,MOYOBAMBA,AG,55.0,FEMENINO,20240108.0,220101.0,295651.0
3,20241203,AREQUIPA,CAYLLOMA,COPORAQUE,AG,50.0,MASCULINO,20230824.0,40506.0,851625.0
4,20241203,LIMA,LIMA,JESUS MARIA,AG,58.0,MASCULINO,20221217.0,150113.0,287786.0


## PASO 2:  Análisis de valores nulos
### Objetivo: Identificar columnas con valores faltantes

In [2]:
# Conteo de nulos
df_covid.isnull().sum()
# Porcentaje de nulos
(df_covid.isnull().sum() / len(df_covid) * 100).round(2)

FECHA_CORTE         0.00
DEPARTAMENTO        0.00
PROVINCIA           0.00
DISTRITO            0.00
METODODX            0.00
EDAD                0.01
SEXO                0.00
FECHA_RESULTADO     0.04
UBIGEO              5.16
id_persona         99.73
dtype: float64

## PASO 3: Análisis de duplicados
### Objetivo: Detectar registros duplicados

In [4]:
# Duplicados completos
duplicados_totales = df_covid.duplicated().sum()
# Duplicados en variables clave
duplicados_clave = df_covid.duplicated(subset=['DEPARTAMENTO', 'EDAD', 'SEXO', 'FECHA_RESULTADO']).sum()
print(f"Duplicados totales: {duplicados_totales}")
print(f"Duplicados en variables clave: {duplicados_clave}")

Duplicados totales: 1109602
Duplicados en variables clave: 3275748


## PASO 4: Crear dataset limpio con conversiones de tipos
### Objetivo: Crear nuevo dataframe con tipos de datos correctos

In [12]:
df_clean_id = df_covid.copy()

# Convertir FECHA_CORTE y FECHA_RESULTADO a datetime
df_clean_id['FECHA_CORTE'] = pd.to_datetime(df_clean_id['FECHA_CORTE'], format='%Y%m%d', errors='coerce')
df_clean_id['FECHA_RESULTADO'] = pd.to_datetime(df_clean_id['FECHA_RESULTADO'], format='%Y%m%d', errors='coerce')

# Convertir EDAD a Int64
df_clean_id['EDAD'] = pd.to_numeric(df_clean_id['EDAD'], errors='coerce').astype('Int64')

# Convertir UBIGEO a string (sin .0)
df_clean_id['UBIGEO'] = pd.to_numeric(df_clean_id['UBIGEO'], errors='coerce').astype('Int64').astype(str)

# Convertir id_persona a string y renombrar a ID_PERSONA
df_clean_id['id_persona'] = pd.to_numeric(df_clean_id['id_persona'], errors='coerce').astype('Int64').astype(str)
df_clean_id.rename(columns={'id_persona': 'ID_PERSONA'}, inplace=True)

# Asignar IDs a registros sin ID_PERSONA
ids_existentes = df_clean_id['ID_PERSONA'].dropna().astype(int)
max_id = ids_existentes.max() if len(ids_existentes) > 0 else 0
filas_sin_id = df_clean_id['ID_PERSONA'].isnull().sum()
nuevos_ids = range(max_id + 1, max_id + 1 + filas_sin_id)
df_clean_id.loc[df_clean_id['ID_PERSONA'].isnull(), 'ID_PERSONA'] = [str(id) for id in nuevos_ids]
df_clean_id.sample(10)

,FECHA_CORTE,DEPARTAMENTO,PROVINCIA,DISTRITO,METODODX,EDAD,SEXO,FECHA_RESULTADO,UBIGEO,ID_PERSONA
1131645,2024-12-03,JUNIN,HUANCAYO,EL TAMBO,AG,47,FEMENINO,2022-01-08,120114,43126814
1475888,2024-12-03,AREQUIPA,AREQUIPA,ALTO SELVA ALEGRE,AG,33,MASCULINO,2022-07-16,40102,43471057
4532705,2024-12-03,UCAYALI,CORONEL PORTILLO,CALLERIA,PR,41,FEMENINO,2020-08-18,250101,46527874
918172,2024-12-03,PUNO,EL COLLAO,ILAVE,AG,42,FEMENINO,2023-03-15,210501,42913341
854616,2024-12-03,LIMA,EN INVESTIGACIÓN,EN INVESTIGACIÓN,PCR,46,FEMENINO,2020-09-08,NaN,42849785
1289989,2024-12-03,SAN MARTIN,RIOJA,RIOJA,AG,22,MASCULINO,2021-10-27,220801,43285158
2532404,2024-12-03,LIMA,LIMA,ATE,AG,25,MASCULINO,2022-01-21,150103,44527573
1553464,2024-12-03,LIMA,LIMA,LIMA,PR,31,FEMENINO,2020-05-21,150101,43548633
2359712,2024-12-03,LIMA,LIMA,SANTA ANITA,PCR,47,FEMENINO,2022-01-24,150137,44354881
4351840,2024-12-03,JUNIN,HUANCAYO,HUANCAYO,PR,50,MASCULINO,2020-11-27,120101,46347009


In [13]:
(df_clean_id.isnull().sum() / len(df_clean_id) * 100).round(2)

FECHA_CORTE        0.00
DEPARTAMENTO       0.00
PROVINCIA          0.00
DISTRITO           0.00
METODODX           0.00
EDAD               0.01
SEXO               0.00
FECHA_RESULTADO    0.04
UBIGEO             5.16
ID_PERSONA         0.00
dtype: float64

## PASO 5: Verificación de variables clave
### Objetivo: Validar rangos y valores únicos

In [14]:
# Estadísticas de edad
df_clean_id['EDAD'].describe()

# Valores únicos en SEXO
df_clean_id['SEXO'].value_counts()

# Departamentos
df_clean_id['DEPARTAMENTO'].nunique()


print(f"Valores únicos en DEPARTAMENTO: {df_clean_id['DEPARTAMENTO'].nunique()}")
print("")
print(f"Valores únicos en {df_clean_id['SEXO'].value_counts()}")
print("")
print(f"Estadísticas de edad:\n{df_clean_id['EDAD'].describe()}")

Valores únicos en DEPARTAMENTO: 25

Valores únicos en SEXO
FEMENINO     2376880
MASCULINO    2208480
Name: count, dtype: int64

Estadísticas de edad:
count    4585007.0
mean     40.791462
std      17.716466
min            0.0
25%           28.0
50%           39.0
75%           53.0
max          125.0
Name: EDAD, dtype: Float64


## PASO 6: Eliminar valores nulos en variables críticas
### Objetivo: Eliminar registros con nulos en EDAD, FECHA_RESULTADO, UBIGEO


In [16]:
# Registros iniciales
print(f"Inicial: {len(df_clean_id):,}")

# Eliminar EDAD nulos
df_clean_id = df_clean_id[df_clean_id['EDAD'].notna()]

# Eliminar FECHA_RESULTADO nulos
df_clean_id = df_clean_id[df_clean_id['FECHA_RESULTADO'].notna()]

# Eliminar UBIGEO nulos (incluyendo 'nan' como string)
df_clean_id = df_clean_id[
    (df_clean_id['UBIGEO'].notna()) & 
    (df_clean_id['UBIGEO'] != 'nan') & 
    (df_clean_id['UBIGEO'] != '<NA>')
]

print(f"Final: {len(df_clean_id):,}")

Inicial: 4,585,360
Final: 4,346,389


## PASO 7: Filtrar fechas válidas (>= 2020)
### Objetivo: Eliminar registros con fechas inválidas anteriores a 2020



In [17]:
# Definir fecha de inicio de COVID-19
fecha_inicio_covid = pd.to_datetime('2020-01-01')

# Filtrar solo fechas >= 2020-01-01
df_clean_id = df_clean_id[df_clean_id['FECHA_RESULTADO'] >= fecha_inicio_covid]

## PASO 8: Validar rangos de edad (0-100 años)
### Objetivo: Eliminar edades inválidas mayores a 100 años

In [18]:
# Filtrar edades válidas (0 a 100 años)
df_clean_id = df_clean_id[(df_clean_id['EDAD'] >= 0) & (df_clean_id['EDAD'] <= 100)]

## PASO 9: Crear grupos de edad
### Objetivo: Categorizar edades en grupos epidemiológicos

In [20]:
# Definir función de categorización
def categorizar_edad(edad):
    if edad < 18:
        return '0-17 años (Menores)'
    elif edad < 30:
        return '18-29 años (Jóvenes)'
    elif edad < 45:
        return '30-44 años (Adultos Jóvenes)'
    elif edad < 60:
        return '45-59 años (Adultos)'
    elif edad < 75:
        return '60-74 años (Adultos Mayores)'
    else:
        return '75+ años (Tercera Edad)'

# Aplicar categorización
df_clean_id['GRUPO_EDAD'] = df_clean_id['EDAD'].apply(categorizar_edad)
df_clean_id['GRUPO_EDAD'].value_counts()

GRUPO_EDAD
30-44 años (Adultos Jóvenes)    1418424
45-59 años (Adultos)             982981
18-29 años (Jóvenes)             948676
60-74 años (Adultos Mayores)     504573
0-17 años (Menores)              309655
75+ años (Tercera Edad)          181226
Name: count, dtype: int64

## PASO 10: Verificación final de calidad
### Objetivo: Confirmar que el dataset está completamente limpio

#### 1. Valores Nulos: 0 nulos en todas las columnas

In [21]:
(df_clean_id.isnull().sum() / len(df_clean_id) * 100).round(2)

FECHA_CORTE        0.0
DEPARTAMENTO       0.0
PROVINCIA          0.0
DISTRITO           0.0
METODODX           0.0
EDAD               0.0
SEXO               0.0
FECHA_RESULTADO    0.0
UBIGEO             0.0
ID_PERSONA         0.0
GRUPO_EDAD         0.0
dtype: float64

#### 2. Verificacion de valores duplicados

In [26]:
# se ejecuto antes y el valor era de 13 
print(f"Registros duplicados: {df_clean_id.duplicated().sum()}")
df_clean_id.drop_duplicates(inplace=True)
print(f"Registros después de eliminar duplicados: {df_clean_id.duplicated().sum()}")

Registros duplicados: 0
Registros después de eliminar duplicados: 0


#### 3.Rangos de edad:  0-100 años (válido)

In [31]:
print(f"Edad mínima: {df_clean_id['EDAD'].min()}")
print(f"Edad máxima: {df_clean_id['EDAD'].max()}")

Edad mínima: 0
Edad máxima: 100


#### 4. Rangos de fechas:  2020-03-06 a 2024-03-04

In [32]:
#rango de fechas
print(f"Fecha mínima: {df_clean_id['FECHA_RESULTADO'].min()}")
print(f"Fecha máxima: {df_clean_id['FECHA_RESULTADO'].max()}")

Fecha mínima: 2020-03-06 00:00:00
Fecha máxima: 2024-03-04 00:00:00


#### 5. Valores categóricos:
- SEXO: 2 valores 
- DEPARTAMENTO: 25 departamentos 
- GRUPO_EDAD: 6 categorías 

In [36]:
# Valores únicos en SEXO
df_clean_id['SEXO'].value_counts()

# Departamentos
df_clean_id['DEPARTAMENTO'].nunique()


print(f"Valores únicos en DEPARTAMENTO: {df_clean_id['DEPARTAMENTO'].nunique()}")
print("")
print(f"Valores únicos en {df_clean_id['SEXO'].value_counts()}")
print("")
#MOSTRAR LAS 6 CATEGORÍAS DE EDAD
df_clean_id['GRUPO_EDAD'].value_counts().head(6)

Valores únicos en DEPARTAMENTO: 25

Valores únicos en SEXO
FEMENINO     2261081
MASCULINO    2084441
Name: count, dtype: int64



GRUPO_EDAD
30-44 años (Adultos Jóvenes)    1418421
45-59 años (Adultos)             982977
18-29 años (Jóvenes)             948671
60-74 años (Adultos Mayores)     504573
0-17 años (Menores)              309655
75+ años (Tercera Edad)          181225
Name: count, dtype: int64

#### 6. Tipos de datos: Todos correctos

In [37]:
df_clean_id.info()

<class 'pandas.DataFrame'>
Index: 4345522 entries, 0 to 4585359
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   FECHA_CORTE      datetime64[us]
 1   DEPARTAMENTO     str           
 2   PROVINCIA        str           
 3   DISTRITO         str           
 4   METODODX         str           
 5   EDAD             Int64         
 6   SEXO             str           
 7   FECHA_RESULTADO  datetime64[us]
 8   UBIGEO           str           
 9   ID_PERSONA       str           
 10  GRUPO_EDAD       str           
dtypes: Int64(1), datetime64[us](2), str(8)
memory usage: 402.0 MB


## PASO 11: Exportar Dataset Limpio

In [38]:
df_clean_id.to_csv('covid_peru_limpio.csv', index=False, sep=';', encoding='utf-8-sig')